# Modeling

Scores are MAPE on time-based folds. `03_eval` covers why those folds.

In [1]:
import os
from pathlib import Path

# The code reads data/... relative to the repo root, and notebooks start in notebooks/.
if Path.cwd().name == "notebooks":
    os.chdir("..")
print(Path.cwd())

/home/indigo/spotterAssessment


### features.py

In [2]:
%%writefile src/spotterassessment/features.py
"""Loading, cleaning, and feature engineering. Nothing here looks at the target,
except bad_rate_mask, which is only ever used to filter training rows and to
separate clean rows when scoring."""
import numpy as np
import pandas as pd

EARTH_RADIUS_MILES = 3958.8
SMOOTH_FEATURES = ["log_distance", "haversine", "weight_clean",
                  "pickup_lat", "pickup_lon", "delivery_lat", "delivery_lon"]


def load(name):
    return pd.read_csv(f"data/{name}.csv", parse_dates=["date"])


def daily_market(*frames):
    """Average market_index per date, used to fill gaps and for the December chart."""
    return pd.concat(frames).groupby("date")["market_index"].mean().rename("market_daily")


def haversine_miles(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return EARTH_RADIUS_MILES * 2 * np.arcsin(np.sqrt(a))


def add_features(df, market_by_date, weight_fill):
    out = df.join(market_by_date, on="date")
    load_market = out["market_index"] if "market_index" in out else out["market_daily"]
    out["market"] = load_market.fillna(out["market_daily"])
    out["weight_clean"] = out["weight"].abs().fillna(weight_fill)
    out["log_distance"] = np.log(out["distance"])
    out["haversine"] = haversine_miles(out["pickup_lat"], out["pickup_lon"],
                                       out["delivery_lat"], out["delivery_lon"])
    out["months_elapsed"] = (out["date"] - pd.Timestamp("2025-01-01")).dt.days / 30
    out["day_of_week"] = out["date"].dt.dayofweek
    return out


def bad_rate_mask(df, low=0.6, high=1.6):
    """True where rate-per-mile is far from similar loads (equipment x distance band)."""
    rate_per_mile = df["posted_rate"] / df["distance"]
    band = pd.qcut(df["distance"], 12, duplicates="drop")
    typical = rate_per_mile.groupby([df["equipment"], band], observed=True).transform("median")
    ratio = rate_per_mile / typical
    return (ratio < low) | (ratio > high)

Writing src/spotterassessment/features.py


### model.py

In [3]:
%%writefile src/spotterassessment/model.py
"""log(rate) = smooth(distance, weight, geography) + equipment + b*market + c*time

The time term is frozen at the last training date. We assume the market term and
the level reached by the end of training carry forward, and do not extrapolate the trend.
"""
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, SplineTransformer

from spotterassessment.features import SMOOTH_FEATURES


class FreightRateModel:
    def __init__(self, alpha=0.01, n_knots=8):
        columns = ColumnTransformer([
            ("smooth", SplineTransformer(n_knots=n_knots, degree=3, extrapolation="constant"), SMOOTH_FEATURES),
            ("equipment", OneHotEncoder(sparse_output=False), ["equipment"]),
            ("level", "passthrough", ["market", "months_elapsed"]),
        ], sparse_threshold=0)
        self.pipeline = make_pipeline(columns, Ridge(alpha=alpha))

    def fit(self, train):
        self.last_time = train["months_elapsed"].max()
        self.pipeline.fit(train, np.log(train["posted_rate"]))
        return self

    def predict(self, df):
        frozen = df.assign(months_elapsed=df["months_elapsed"].clip(upper=self.last_time))
        return np.exp(self.pipeline.predict(frozen))

    @property
    def market_effect(self):
        return self.pipeline[-1].coef_[-2]

    @property
    def monthly_trend(self):
        return self.pipeline[-1].coef_[-1]

Writing src/spotterassessment/model.py


### backtest.py

In [4]:
%%writefile src/spotterassessment/backtest.py
"""Rolling-origin backtest. Every test window is two months, like Nov-Dec."""
import lightgbm as lgb
import numpy as np
import pandas as pd

from spotterassessment.features import add_features, bad_rate_mask, daily_market, load
from spotterassessment.model import FreightRateModel

# name: (last training date, test start, test end)
FORWARD_FOLDS = {
    "Mar-Apr": ("2025-02-28", "2025-03-01", "2025-04-30"),
    "May-Jun": ("2025-04-30", "2025-05-01", "2025-06-30"),
    "Jul-Aug": ("2025-06-30", "2025-07-01", "2025-08-31"),
    "Sep-Oct": ("2025-08-31", "2025-09-01", "2025-10-31"),
}
TREE_FEATURES = ["log_distance", "equipment_code", "weight_clean", "day_of_week",
                 "pickup_lat", "pickup_lon", "delivery_lat", "delivery_lon"]
TREE_PARAMS = dict(objective="l1", n_estimators=600, learning_rate=0.05, num_leaves=63,
                   min_child_samples=30, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                   random_state=0, verbose=-1, deterministic=True, force_row_wise=True, n_jobs=8)


def make_splits(raw, market, low=0.6, high=1.6):
    """Yield (name, train, test). August also runs leave-one-month-out, because it is the
    only labeled month where quote_signal is noise, as it appears to be in Nov-Dec."""
    def build(past, test_rows):
        fill = past["weight"].abs().median()
        train = add_features(past[~bad_rate_mask(past, low, high)], market, fill)
        test = add_features(test_rows.assign(bad_rate=bad_rate_mask(test_rows)), market, fill)
        return train, test

    for name, (cutoff, start, end) in FORWARD_FOLDS.items():
        yield (name, *build(raw[raw["date"] <= cutoff], raw[raw["date"].between(start, end)]))
    august = raw["date"].dt.month == 8
    yield ("Leave-Aug", *build(raw[~august], raw[august]))


def score(test, pred):
    actual = test["posted_rate"].to_numpy()
    clean = ~test["bad_rate"].to_numpy()
    error = actual - pred
    return {
        "mape": (np.abs(error) / actual)[clean].mean() * 100,
        "mae": np.abs(error)[clean].mean(),
        "rmse": np.sqrt((error ** 2)[clean].mean()),
        "mae_all": np.abs(error).mean(),
        "rmse_all": np.sqrt((error ** 2).mean()),
        "bias": (-error / actual)[clean].mean() * 100,
    }


def baseline_predict(train, test):
    bands = np.unique(np.r_[0, train["distance"].quantile(np.linspace(0.1, 0.9, 9)), np.inf])
    rpm = train["posted_rate"] / train["distance"]
    table = rpm.groupby([train["equipment"], pd.cut(train["distance"], bands)], observed=True).median()
    keys = pd.MultiIndex.from_arrays([test["equipment"], pd.cut(test["distance"], bands)])
    return keys.map(table.to_dict()).to_numpy(dtype=float) * test["distance"].to_numpy()


def tree_predict(train, test, features=TREE_FEATURES):
    codes = {"Dry Van": 0, "Flatbed": 1, "Reefer": 2}
    train = train.assign(equipment_code=train["equipment"].map(codes))
    test = test.assign(equipment_code=test["equipment"].map(codes))
    model = lgb.LGBMRegressor(**TREE_PARAMS).fit(train[features], np.log(train["posted_rate"]))
    return np.exp(model.predict(test[features]))


def ridge_predict(train, test):
    return FreightRateModel().fit(train).predict(test)


def evaluate(models, splits):
    rows = [{"model": name, "fold": fold, **score(test, fn(train, test))}
            for fold, train, test in splits for name, fn in models.items()]
    return pd.DataFrame(rows)


def summarize(results):
    forward = results[results["fold"] != "Leave-Aug"]
    summary = forward.groupby("model", sort=False)[["mape", "mae", "rmse", "mae_all", "rmse_all", "bias"]].mean()
    summary["worst_mape"] = forward.groupby("model")["mape"].max()
    folds = results.pivot(index="model", columns="fold", values="mape")
    return summary.join(folds[[*FORWARD_FOLDS, "Leave-Aug"]]).round(2)


def main():
    raw, future = load("train_test"), load("validation")
    splits = list(make_splits(raw, daily_market(raw, future)))
    models = {"median baseline": baseline_predict, "first LightGBM": tree_predict, "structural ridge": ridge_predict}
    pd.set_option("display.width", 200, "display.max_columns", 20)
    print(summarize(evaluate(models, splits)).to_string())


if __name__ == "__main__":
    main()

Writing src/spotterassessment/backtest.py


### predict.py

In [5]:
%%writefile src/spotterassessment/predict.py
"""Train on all labeled data and write both submission files."""
import pandas as pd

from spotterassessment.features import add_features, bad_rate_mask, daily_market, load
from spotterassessment.model import FreightRateModel


def main():
    train = load("train_test")
    validation = load("validation")
    december = load("december_chart_inputs")
    template = pd.read_csv("data/validation_predictions_template.csv")

    market = daily_market(train, validation)
    fill = train["weight"].abs().median()
    model = FreightRateModel().fit(add_features(train[~bad_rate_mask(train)], market, fill))
    print(f"market effect {model.market_effect:.4f}, monthly trend {model.monthly_trend:.4f}")

    predictions = model.predict(add_features(validation, market, fill)).round(2)
    template["predicted_rate"] = template["load_id"].map(dict(zip(validation["load_id"], predictions)))
    template.to_csv("validation_predictions.csv", index=False)

    # The December file has no coordinates, so look them up from training rows.
    pickup = train.loc[train["pickup"] == "Lexington", ["pickup_lat", "pickup_lon"]].iloc[0]
    delivery = train.loc[train["delivery"] == "Fort Wayne", ["delivery_lat", "delivery_lon"]].iloc[0]
    rows = add_features(december.assign(**pickup, **delivery), market, fill)
    december["predicted_rate"] = model.predict(rows).round(2)
    december.to_csv("data/december_chart_inputs.csv", index=False)


if __name__ == "__main__":
    main()

Writing src/spotterassessment/predict.py


### experiments.py

In [6]:
%%writefile src/spotterassessment/experiments.py
"""Reproduces every rejected-idea comparison quoted in the report."""
import numpy as np
import pandas as pd

from spotterassessment.backtest import (TREE_FEATURES, evaluate, make_splits, ridge_predict, score,
                                        summarize, tree_predict)
from spotterassessment.features import add_features, bad_rate_mask, daily_market, load
from spotterassessment.model import FreightRateModel


class DollarRateModel(FreightRateModel):
    """The same model, fitted on dollars instead of log(rate)."""

    def fit(self, train):
        self.last_time = train["months_elapsed"].max()
        self.pipeline.fit(train, train["posted_rate"])
        return self

    def predict(self, df):
        frozen = df.assign(months_elapsed=df["months_elapsed"].clip(upper=self.last_time))
        return self.pipeline.predict(frozen)


def main():
    raw, future = load("train_test"), load("validation")
    market = daily_market(raw, future)
    splits = list(make_splits(raw, market))
    pd.set_option("display.width", 220, "display.max_columns", 20)

    print("== Log rate vs dollars ==")
    dollars = lambda tr, te: DollarRateModel().fit(tr).predict(te)
    print(summarize(evaluate({"log rate": ridge_predict, "dollars": dollars}, splits)).to_string())
    print("non-positive dollar predictions:", sum((dollars(tr, te) <= 0).sum() for _, tr, te in splits))

    print("== Trees and the trap features ==")
    tree_with = lambda extra: lambda tr, te: tree_predict(tr, te, TREE_FEATURES + extra)
    print(summarize(evaluate({
        "tree": tree_predict,
        "tree + quote_signal": tree_with(["quote_signal"]),
        "tree + market_index": lambda tr, te: tree_predict(tr.assign(mi=tr["market"]), te.assign(mi=te["market"]), TREE_FEATURES + ["mi"]),
    }, splits)).to_string())

    print("== Regularization strength ==")
    print(summarize(evaluate({f"alpha={a}": (lambda a: lambda tr, te: FreightRateModel(alpha=a).fit(tr).predict(te))(a)
                              for a in (0.001, 0.01, 0.1, 1.0, 10.0)}, splits)).to_string())

    print("== Time and market terms ==")
    def variant(no_market=False, no_time=False, extrapolate=False):
        def fit_predict(tr, te):
            tr2, te2 = tr.copy(), te.copy()
            if no_market:
                tr2["market"], te2["market"] = 0.0, 0.0
            if no_time:
                tr2["months_elapsed"], te2["months_elapsed"] = 0.0, 0.0
            model = FreightRateModel().fit(tr2)
            if extrapolate:
                model.last_time = np.inf
            return model.predict(te2)
        return fit_predict
    print(summarize(evaluate({
        "final: market + frozen trend": variant(),
        "no market, no time": variant(no_market=True, no_time=True),
        "market only": variant(no_time=True),
        "market + extrapolated trend": variant(extrapolate=True),
    }, splits)).to_string())

    print("== Corrupted-rate threshold ==")
    rows = []
    for low, high in [(0, 1e9), (0.4, 2.5), (0.5, 2.0), (0.6, 1.6), (0.7, 1.4), (0.8, 1.25)]:
        result = evaluate({f"keep {low}-{high}": ridge_predict}, list(make_splits(raw, market, low, high)))
        rows.append(result)
    print(summarize(pd.concat(rows)).to_string())

    print("== Random split vs time split (ridge and tree + quote_signal) ==")
    fill = raw["weight"].abs().median()
    full = add_features(raw.assign(bad_rate=bad_rate_mask(raw)), market, fill)
    in_train = np.random.RandomState(0).rand(len(full)) < 0.8
    tr, te = full[in_train & ~full["bad_rate"]], full[~in_train]
    for name, fn in {"tree": tree_predict, "tree + quote_signal": tree_with(["quote_signal"]),
                     "ridge": ridge_predict}.items():
        print(name, {k: round(v, 2) for k, v in score(te, fn(tr, te)).items()})

    print("== Held-out cities: 8 random cities, 10 repeats, test Sep-Oct ==")
    cities = sorted(set(raw["pickup"]) | set(raw["delivery"]))
    rng = np.random.RandomState(1)
    rows = []
    for repeat in range(10):
        held = set(rng.choice(cities, 8, replace=False))
        touches = full["pickup"].isin(held) | full["delivery"].isin(held)
        tr = full[(full["date"] <= "2025-08-31") & ~full["bad_rate"] & ~touches]
        te = full[(full["date"] > "2025-08-31") & touches]
        rows.append({"model": "ridge", **score(te, ridge_predict(tr, te))})
        rows.append({"model": "tree", **score(te, tree_predict(tr, te))})
    print(pd.DataFrame(rows).groupby("model")[["mape", "mae", "bias"]].agg(["mean", "std"]).round(2).to_string())

    print("== Extreme cities held out: Sep-Oct ==")
    latitude = full.groupby("pickup")["pickup_lat"].first().sort_values()
    longitude = full.groupby("pickup")["pickup_lon"].first().sort_values()
    groups = {"south": latitude.index[:3], "north": latitude.index[-2:], "west": longitude.index[:2], "east": longitude.index[-2:]}
    for name, members in groups.items():
        touches = full["pickup"].isin(members) | full["delivery"].isin(members)
        tr = full[(full["date"] <= "2025-08-31") & ~full["bad_rate"] & ~touches]
        te = full[(full["date"] > "2025-08-31") & touches]
        print(name, list(members), "ridge", round(score(te, ridge_predict(tr, te))["mape"], 2),
              "tree", round(score(te, tree_predict(tr, te))["mape"], 2))


if __name__ == "__main__":
    main()

Writing src/spotterassessment/experiments.py


## Load data

In [7]:
import numpy as np
import pandas as pd

from spotterassessment.backtest import (TREE_FEATURES, baseline_predict, evaluate, make_splits,
                                        ridge_predict, summarize, tree_predict)
from spotterassessment.experiments import DollarRateModel
from spotterassessment.features import add_features, bad_rate_mask, daily_market, load
from spotterassessment.model import FreightRateModel

pd.set_option("display.width", 220, "display.max_columns", 20)
raw, future = load("train_test"), load("validation")
market = daily_market(raw, future)
fill = raw["weight"].abs().median()
splits = list(make_splits(raw, market))

## Baseline and LightGBM

In [8]:
first_models = {"median baseline": baseline_predict, "first LightGBM": tree_predict}
print(summarize(evaluate(first_models, splits)).to_string())

                 mape     mae    rmse  mae_all  rmse_all  bias  worst_mape  Mar-Apr  May-Jun  Jul-Aug  Sep-Oct  Leave-Aug
model                                                                                                                    
median baseline  4.73  108.26  147.06   158.63    622.66 -2.89        6.17     5.26     6.17     3.80     3.68       3.71
first LightGBM   3.76   91.59  125.14   142.04    617.67 -2.90        5.87     4.67     5.87     2.42     2.08       2.06


LightGBM beats the baseline, but both underprice by about 2.9%. That looks like a problem with the level of rates over time.

## LightGBM with `market_index`

In [9]:
print(summarize(evaluate({
    "tree": tree_predict,
    "tree + market_index": lambda tr, te: tree_predict(tr.assign(mi=tr["market"]), te.assign(mi=te["market"]), TREE_FEATURES + ["mi"]),
}, splits)).to_string())

                     mape    mae    rmse  mae_all  rmse_all  bias  worst_mape  Mar-Apr  May-Jun  Jul-Aug  Sep-Oct  Leave-Aug
model                                                                                                                       
tree                 3.76  91.59  125.14   142.04    617.67 -2.90        5.87     4.67     5.87     2.42     2.08       2.06
tree + market_index  3.23  77.12  108.30   127.82    613.40 -2.75        4.93     2.93     2.65     2.43     4.93       1.77


This helps the early folds but breaks Sep-Oct (4.93%). The tree seems to use the market as a stand-in for time of year.

## Final model

`log(rate) = smooth(distance, weight, coordinates) + equipment + market + time`

A ridge regression with splines. The time term stops at the last training date, so the trend isn't projected forward.

Log rate vs dollars:

In [10]:
# Log rate vs dollars
dollars = lambda tr, te: DollarRateModel().fit(tr).predict(te)
print(summarize(evaluate({"log rate": ridge_predict, "dollars": dollars}, splits)).to_string())
print("non-positive dollar predictions:", sum((dollars(tr, te) <= 0).sum() for _, tr, te in splits))

          mape    mae    rmse  mae_all  rmse_all  bias  worst_mape  Mar-Apr  May-Jun  Jul-Aug  Sep-Oct  Leave-Aug
model                                                                                                            
log rate  2.38  57.16   83.40   108.11    605.45 -0.45        3.00     2.21     2.07     3.00     2.24       1.90
dollars   5.30  90.16  127.56   140.63    613.54  0.63        5.74     4.79     5.25     5.74     5.41       4.95
non-positive dollar predictions: 4


Log wins, 2.38% to 5.30%, and the dollar version produces some non-positive predictions.

Fit on all clean rows and check that predictions move sensibly:

In [11]:
# The final model, trained on all clean labeled rows
all_train = add_features(raw[~bad_rate_mask(raw)], market, fill)
final = FreightRateModel().fit(all_train)
print(f"market effect {final.market_effect:.3f}, trend per 30 days {final.monthly_trend:.4f}")

# Sanity checks: change one input at a time
base = dict(distance=1000., equipment="Dry Van", weight=32000., date=pd.Timestamp("2025-12-03"),
            pickup_lat=36.99, pickup_lon=-85., delivery_lat=41.3, delivery_lon=-85.36, market_index=.95)

def curve(column, values):
    rows = pd.DataFrame([{**base, column: value} for value in values])
    return dict(zip(values, final.predict(add_features(rows, market, fill)).round(0)))

print("distance", curve("distance", [100, 250, 500, 1000, 2000, 3000]))
print("weight", curve("weight", [5000, 20000, 30000, 40000, 47500]))
print("equipment", curve("equipment", ["Dry Van", "Flatbed", "Reefer"]))

validation_rows = add_features(future, market, fill)
validation_pred = final.predict(validation_rows)
print("validation predictions:", validation_pred.min().round(0), "to", validation_pred.max().round(0),
      "| non-positive:", (validation_pred <= 0).sum())

market effect 0.142, trend per 30 days 0.0063
distance {100: np.float64(286.0), 250: np.float64(621.0), 500: np.float64(1103.0), 1000: np.float64(2023.0), 2000: np.float64(3753.0), 3000: np.float64(5411.0)}
weight {5000: np.float64(1918.0), 20000: np.float64(1938.0), 30000: np.float64(2005.0), 40000: np.float64(2071.0), 47500: np.float64(2081.0)}
equipment {'Dry Van': np.float64(2023.0), 'Flatbed': np.float64(2194.0), 'Reefer': np.float64(2285.0)}
validation predictions: 194.0 to 6904.0 | non-positive: 0
